# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library. All dataset entities (record sets, fields, columns) are referenced by their `@id` attributes, ensuring clarity and consistency.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```


In [ ]:
# Ensure `mlcroissant` and required libraries are installed
!pip install mlcroissant pandas matplotlib --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

The `mlcroissant.Dataset` object allows accessing metadata (schema, description, etc.) and iterating over records from each record set. We will display the descriptive metadata from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
from pprint import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display basic metadata
metadata = dataset.metadata
print('\nDataset Title:')
print(metadata.name)
print('\nDataset Description:')
print(metadata.description)

# Show keywords and identifier
print('\nKeywords:', getattr(metadata, 'keywords', 'N/A'))
print('DOI Identifier:', getattr(metadata, 'identifier', 'N/A'))
print('Published:', getattr(metadata, 'datePublished', 'N/A'))

## 2. Data Overview
Review available record sets and their fields using their `@id`. All further dataset manipulation references these `@id` fields.

**Note:** Since the Croissant metadata structure may vary, we use `dataset.metadata.recordSets` to enumerate record sets, and for each one, enumerate fields using their `@id`. If record sets or fields are missing, a fallback will be displayed.

In [ ]:
# Display all available record sets and their fields (by `@id`)
record_sets = getattr(metadata, 'recordSets', [])
if not record_sets:
    # If no recordSets, try the legacy attribute
    record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print('No record sets found in metadata.')
else:
    print('Record Sets and their fields:')
    for rs in record_sets:
        rs_id = getattr(rs, '@id', None)
        rs_name = getattr(rs, 'name', 'Unnamed RecordSet')
        print(f'- Record set: {rs_name} (@id={rs_id})')
        fields = getattr(rs, 'fields', [])
        if not fields:
            fields = getattr(rs, 'field', [])
        if fields:
            for f in fields:
                field_id = getattr(f, '@id', None)
                field_name = getattr(f, 'name', 'Unnamed Field')
                print(f'   - Field: {field_name} (@id={field_id})')
        else:
            print('   (No fields found for record set)')

# Example: List first 2 records from each record set
for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    print(f'\nSample records from record set {rs_id}:')
    # Use the `@id` for record_set argument
    try:
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            pprint(rec)
            if i >= 1:
                break
    except Exception as e:
        print(f'Error reading records for {rs_id}:', e)

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use each record set and field `@id` for referencing.

**The following code loads all records for each record set into a pandas DataFrame, keyed by their `@id`. Columns correspond to the field `@id`s if present.**

In [ ]:
# Extract records from each record set, referencing them by their `@id`
dataframes = {}
record_set_ids = []

for rs in record_sets:
    rs_id = getattr(rs, '@id', None)
    record_set_ids.append(rs_id)
    try:
        recs = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f'\nRecord set (@id={rs_id}) DataFrame columns: {df.columns.tolist()}')
        print(df.head())
    except Exception as e:
        print(f'Error loading record set {rs_id}:', e)

# Pick the first available record set for demonstration
if record_set_ids:
    sample_record_set_id = record_set_ids[0]
    print('\nAvailable record set IDs:', record_set_ids)
    print(f'\nFields in sample record set (@id={sample_record_set_id}):')
    print(dataframes[sample_record_set_id].columns.tolist())
    print(dataframes[sample_record_set_id].head())
else:
    print('No record sets available for extraction.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter numeric fields, normalize, group, etc., referenced by their `@id`.

**Adjust the following EDA section based on the record set and field `@id`s extracted above.**

In [ ]:
# Choose the sample record set and a numeric field for analysis
record_set_id = sample_record_set_id
df = dataframes[record_set_id]

# List all available columns (fields):
print('Available fields:', df.columns.tolist())

# Try to infer a numeric field by examining the dtypes
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if not numeric_fields:
    # If no numeric fields auto-detected, try an example field name
    numeric_field = df.columns[0]  # Substitute with a field that looks numeric
else:
    numeric_field = numeric_fields[0]

print(f'Selected numeric field (@id={numeric_field}) for EDA.')

# Filtering and normalization
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f'\nFiltered records with {numeric_field} > {threshold}:')
print(filtered_df.head())

# Normalization
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by another field
group_fields = [col for col in df.columns if col != numeric_field]
if group_fields:
    group_field = group_fields[0]
    print(f'\nGrouping by field (@id={group_field})')
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print('\nGrouped mean of numeric field:')
        print(grouped_df.head())
else:
    print('No group fields available.')

## 5. Visualization
Visualize distributions or relationships between fields in the dataset using their `@id`.

**This cell demonstrates a histogram and a boxplot of the selected numeric field. Adjust field names to match those actually found in the record set's DataFrame.**

In [ ]:
# Visualize the numeric field distributions
plt.figure(figsize=(8, 4))
df[numeric_field].hist(bins=10)
plt.title(f'Distribution of {numeric_field} (@id={numeric_field})')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

plt.figure(figsize=(6, 4))
df.boxplot(column=numeric_field)
plt.title(f'Boxplot of {numeric_field} (@id={numeric_field})')
plt.ylabel(numeric_field)
plt.show()

## 6. Conclusion
This notebook showcased loading and exploration of the FAIR² dataset using `mlcroissant`, referencing all entities by their `@id`. Key steps included:

- Displaying metadata and overview of record sets and fields
- Extracting records to pandas DataFrames by record set `@id`
- Applying basic EDA operations: filtering, normalization, grouping
- Visualizing a numeric field distribution

For more advanced analysis, consider referencing additional fields or combining information across multiple record sets using their `@id`. Note that due to small sample size, statistical insights may be limited.